# Analysis of Connection Points

## Introduction

This section demonstrates how to handle and interpolate frames from CSV files containing body joint coordinates.

### Loading and Sorting CSV Files
First, we define functions to sort, load, and preprocess the CSV files.

```python

In [3]:
import os
import re
import numpy as np
import pandas as pd

def sort_key(filename):
    match = re.search(r'keypoints_frame_(\d+)\.csv', filename)
    if match:
        return int(match.group(1))
    return filename

def load_csv_files(directory):
    csv_files = sorted([f for f in os.listdir(directory) if f.endswith('.csv')], key=sort_key)
    data_frames = [pd.read_csv(os.path.join(directory, f)) for f in csv_files]
    return data_frames

### Interpolating Frames

Next, we interpolate the frames to ensure we have a consistent number of frames (e.g., 30 frames) for further analysis.

In [27]:
def interpolate_frames(data_frames, num_frames=30):
    original_frames = len(data_frames)
    interpolated_frames = []
    
    for i in range(num_frames):
        alpha = i * (original_frames - 1) / (num_frames - 1)
        lower_index = int(np.floor(alpha))
        upper_index = int(np.ceil(alpha))
        
        if lower_index == upper_index:
            interpolated_frames.append(data_frames[lower_index])
        else:
            lower_frame = data_frames[lower_index].copy()
            upper_frame = data_frames[upper_index]
            
            weight = alpha - lower_index
            
            for column in ['x', 'y']:
                lower_values = lower_frame[column].values
                upper_values = upper_frame[column].values
                
                # Handle 0 values by replacing them with NaN
                lower_values = np.where(lower_values == 0, np.nan, lower_values)
                upper_values = np.where(upper_values == 0, np.nan, upper_values)
                
                # Interpolate the values, skipping NaNs
                interpolated_values = np.where(
                    np.isnan(lower_values) & np.isnan(upper_values),
                    0,  # If both are NaN, set the result to 0
                    np.where(
                        np.isnan(lower_values),
                        upper_values,  # If lower is NaN, take upper
                        np.where(
                            np.isnan(upper_values),
                            lower_values,  # If upper is NaN, take lower
                            lower_values * (1 - weight) + upper_values * weight  # Both are present
                        )
                    )
                )
                
                lower_frame[column] = interpolated_values
            
            # Interpolating confidence values
            lower_confidence = lower_frame['z'].values
            upper_confidence = upper_frame['z'].values
            
            interpolated_confidence = np.where(
                np.isnan(lower_confidence) & np.isnan(upper_confidence),
                0,  # If both are NaN, set the result to 0
                np.where(
                    np.isnan(lower_confidence),
                    upper_confidence,  # If lower is NaN, take upper
                    np.where(
                        np.isnan(upper_confidence),
                        lower_confidence,  # If upper is NaN, take lower
                        lower_confidence * (1 - weight) + upper_confidence * weight  # Both are present
                    )
                )
            )
            
            lower_frame['z'] = interpolated_confidence
            
            interpolated_frames.append(lower_frame)
    
    return interpolated_frames


### Saving Interpolated Frames
Finally, we save the interpolated frames back to CSV files.

In [4]:
def save_csv_files(data_frames, output_directory):
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
    for i, df in enumerate(data_frames):
        df.to_csv(os.path.join(output_directory, f'frame_{i:03d}.csv'), index=False)

### Processing CSV Files
We process the CSV files from the input directory, interpolate them, and save the results to the output directory.

In [ ]:
# Directory containing the original CSV files
input_directory = 'Dataset/output_2'
# Directory where the new CSV files will be saved
output_directory = 'Dataset/output_2_30f'

for pen in os.listdir(input_directory):
    if pen.startswith('penalty_'):
        print(pen)
        full_path = os.path.join(input_directory, pen)
        output_path = os.path.join(output_directory, pen)
        # Load CSV files
        data_frames = load_csv_files(full_path)
        # Interpolate frames to get 30 files
        interpolated_frames = interpolate_frames(data_frames, num_frames=30)
        # Save the new CSV files
        save_csv_files(interpolated_frames, output_path)

